# Lesson 19: Real-World Case Studies & Safe Write-Capable Servers

In this notebook, you will:
1. Build a `FastMCP` **Write-Capable Social Publishing** server with opaque IDs, idempotency keys, and `playground` dry-runs.
2. Build a **Week-by-Week Study Plan Generator** tool paired with `gemini-2.5-flash` (with offline fallback).

In [ ]:
!pip install -q -U "mcp<2" google-genai

In [ ]:
from mcp.server.fastmcp import FastMCP
from mcp.types import ToolAnnotations

mcp = FastMCP("SafePublisherAndStudyPlanner")
POSTS = {}

@mcp.tool(annotations=ToolAnnotations(readOnlyHint=False, destructiveHint=True, idempotentHint=True))
def create_post(text: str, platforms: list[str], idempotency_key: str) -> dict:
    """Draft a social post or dry-run using platforms=['playground']."""
    if idempotency_key in POSTS:
        return POSTS[idempotency_key]
    status = "discarded_noop" if platforms == ["playground"] else "draft"
    record = {"id": f"p_{len(POSTS)+1}", "status": status, "platforms": platforms, "text": text}
    POSTS[idempotency_key] = record
    return record

@mcp.tool(annotations=ToolAnnotations(readOnlyHint=True))
def generate_study_plan(topic: str, weeks: int = 4) -> list[dict]:
    """Generate a structured week-by-week study roadmap."""
    return [{"week": w, "focus": f"{topic} - Module {w}"} for w in range(1, weeks + 1)]

print("Dry-run post:", create_post("MCP 2026-07-28 is live!", ["playground"], "key-001"))
print("Study plan:", generate_study_plan("MCP Engineering", 3))

### Challenge Exercise
Add an `approve_and_schedule(post_id: str, iso_time: str)` tool that refuses to schedule any post that is not currently in `status == 'draft'`.